In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_meeting_calendly AS (   
WITH coaching_clients AS (
    SELECT id AS lead_id
    FROM crm_ingestion.silver.leads_processed
    WHERE status_label = 'Coaching Client'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
calendly_events AS (
    SELECT
        INVITEE_NAME,
        EVENT_START_TIME
    FROM crm_ingestion.silver.calendly_scheduled_events_processed
    WHERE INVITEE_NAME IS NOT NULL
      AND EVENT_START_TIME IS NOT NULL
      AND DATE(EVENT_START_TIME) < CURRENT_DATE
    QUALIFY ROW_NUMBER() OVER (PARTITION BY EVENT_URI ORDER BY INVITEE_UPDATED_AT DESC) = 1
),
contacts_bridge AS (
    SELECT DISTINCT
        lc.name,
        lc.leads_processed_id AS lead_id
    FROM crm_ingestion.silver.leads_processed_contacts lc
    WHERE lc.name IS NOT NULL
),
calendly_matched AS (
    SELECT
        cb.lead_id,
        ce.EVENT_START_TIME
    FROM contacts_bridge cb
    JOIN calendly_events ce ON ce.INVITEE_NAME = cb.name
    QUALIFY ROW_NUMBER() OVER (PARTITION BY cb.lead_id ORDER BY ce.EVENT_START_TIME DESC) = 1
)
SELECT
    cc.lead_id,
    DATEDIFF(CURRENT_DATE, DATE(cm.EVENT_START_TIME)) AS days_since_last_meeting_calendly
FROM coaching_clients cc
JOIN calendly_matched cm ON cc.lead_id = cm.lead_id
)